# 3. Dictionnaire 
3.1 Inventaire · 3.2 Planning · 3.3 Distances · 3.4 Grille tarifaire · 3.5 Cartographie (fichier à part) · 3.6 Données calculées



In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", None)
DATA_DIR = Path("../docs")  # même dossier que le notebook Milestone 1
SCHEDULE, DISTANCES, RATES = DATA_DIR / "released_schedule.csv", DATA_DIR / "distances.csv", DATA_DIR / "rate_sheet.csv"

# Les CSV commencent par un BOM : "utf-8-sig" le retire.
schedule = pd.read_csv(SCHEDULE, encoding="utf-8-sig", parse_dates=["date", "transmission_date", "version_date"])
schedule = schedule[schedule["date"] < "2027-01-01"]  # 01/01/2027 écarté (usine fermée fin décembre, qualité §3.2)
distances = pd.read_csv(DISTANCES, encoding="utf-8-sig")
grille = pd.read_csv(RATES, dtype={"km_up_to": str}).set_index("km_up_to")  # grille du PDF, transcrite en CSV
besoins = schedule[schedule["schedule_quantity"] > 0]  # besoins réels (qualité §1)
CLE_SCHEDULE = ["date", "supplier_id", "material", "plant", "unloading_point"]

## 3.1 Inventaire des sources

| | `released_schedule.csv` | `distances.csv` | `rate_sheet.pdf` |
|---|---|---|---|
| Rôle | Besoin : quoi, quand, où, combien | Distance → ligne de la grille | Prix d'un envoi (= un camion) ; transcrit dans `rate_sheet.csv` |
| Une ligne = | fournisseur × article × usine × quai × date | fournisseur | tranche de km × 15 tranches de kg |
| Clé | 5 colonnes de `CLE_SCHEDULE` | `supplier_id` | (distance, poids) |
| Origine, mise à jour | *programme usine, à chaque version* | *référentiel, rarement* | *contrat transporteur, à chaque renégociation* |

In [2]:
def a_un_bom(chemin):
    with open(chemin, "rb") as f:
        return f.read(3) == b"\xef\xbb\xbf"

pd.DataFrame({
    "released_schedule.csv": [len(schedule), schedule.shape[1], a_un_bom(SCHEDULE),
                              f"{besoins['date'].min():%d/%m/%Y} → {besoins['date'].max():%d/%m/%Y}"],
    "distances.csv": [len(distances), distances.shape[1], a_un_bom(DISTANCES), "non datée"],
}, index=["lignes", "colonnes", "BOM", "période des besoins"])

,released_schedule.csv,distances.csv
lignes,5308,156
colonnes,10,2
BOM,True,True
période des besoins,09/11/2026 → 18/12/2026,non datée


## 3.2 Dictionnaire de `released_schedule.csv`
Statistiques sur les besoins réels (quantité > 0).

In [3]:
definitions = pd.DataFrame([
    ("date", "jour", "Jour où la pièce doit être à l'usine", "Jamais le week-end. 11/11 férié : livré avant"),
    ("supplier_id", "–", "Fournisseur", "Clé vers distances.csv"),
    ("material", "–", "Référence de la pièce", "Parfois livrée à plusieurs usines"),
    ("plant", "–", "Usine de destination", "Fournisseur × usine = couple, unité de consolidation"),
    ("unloading_point", "–", "Quai de déchargement", "1 quai = 1 usine ; plusieurs quais par camion (confirmé)"),
    ("schedule_quantity", "pièces", "Pièces à livrer", "Entiers ; 0 = pas de besoin (qualité §1)"),
    ("weight", "kg", "Poids de la ligne, emballage compris (confirmé)", "Chaque ligne a son propre emballage : part fixe + poids par pièce (qualité §4) ; manquants (qualité §5)"),
    ("volume", "m³", "Volume du conteneur de la ligne (confirmé)", "Fixe par article × quai, indépendant de la quantité (3.2.1)"),
    ("transmission_date", "jour", "Date d'envoi du programme au fournisseur (définition du cadrage)", "Non utilisée"),
    ("version_date", "jour", "Version du programme", "Une seule version ; non utilisée"),
], columns=["colonne", "unité", "description", "remarque"]).set_index("colonne")

stats = pd.DataFrame({
    "manquants (toutes lignes)": schedule.isna().sum(),
    "manquants (besoins)": besoins.isna().sum(),
    "distinctes": besoins.nunique(),
    "min": besoins.min(numeric_only=False),
    "max": besoins.max(numeric_only=False),
})
for c in ["min", "max"]:
    stats[c] = stats[c].map(lambda v: f"{v:%d/%m/%Y}" if isinstance(v, pd.Timestamp) else v)
definitions.join(stats)

,unité,description,remarque,manquants (toutes lignes),manquants (besoins),distinctes,min,max
colonne,,,,,,,,
date,jour,Jour où la pièce doit être à l'usine,Jamais le week-end. 11/11 férié : livré avant,0,0,30,09/11/2026,18/12/2026
supplier_id,–,Fournisseur,Clé vers distances.csv,0,0,93,SUP_00001,SUP_00156
material,–,Référence de la pièce,Parfois livrée à plusieurs usines,0,0,1110,MAT_000001,MAT_001229
plant,–,Usine de destination,"Fournisseur × usine = couple, unité de consolidation",0,0,4,PLT_001,PLT_004
unloading_point,–,Quai de déchargement,1 quai = 1 usine ; plusieurs quais par camion (confirmé),0,0,49,ULP_0001,ULP_0050
schedule_quantity,pièces,Pièces à livrer,Entiers ; 0 = pas de besoin (qualité §1),0,0,367,1.0,100000.0
weight,kg,"Poids de la ligne, emballage compris (confirmé)",Chaque ligne a son propre emballage : part fixe + poids par pièce (qualité §4) ; manquants (qualité §5),163,134,1546,0.77,26905.2
volume,m³,Volume du conteneur de la ligne (confirmé),"Fixe par article × quai, indépendant de la quantité (3.2.1)",143,116,59,0.00882,5.808
transmission_date,jour,Date d'envoi du programme au fournisseur (définition du cadrage),Non utilisée,0,0,12,02/02/2026,07/11/2026


### 3.2.1 Preuves des remarques propres au dictionnaire
Le modèle du poids, les manquants et les doublons sont prouvés dans le notebook qualité (§4, §5, §2) ; les 24 cas atypiques du poids sont listés ci-dessous. Ici : la structure, et la cohérence de `volume` avec un conteneur par ligne (confirmé par le métier).

In [4]:
cle_vol = ["material", "plant", "unloading_point"]
vol = besoins.dropna(subset=["volume"]).groupby(cle_vol)["volume"].nunique()

print("Besoins le week-end :", int((besoins["date"].dt.dayofweek >= 5).sum()))
print("Besoins le 11/11 (férié) :", int((besoins["date"] == "2026-11-11").sum()))
print("Usines max par quai :", besoins.groupby("unloading_point")["plant"].nunique().max())
print("Articles livrés à plusieurs usines :", int((besoins.groupby("material")["plant"].nunique() > 1).sum()),
      "/", besoins["material"].nunique())
print("Quantités entières :", bool((schedule["schedule_quantity"] % 1 == 0).all()))
print("Article × quai dont le volume varie :", int((vol > 1).sum()), "/", len(vol))
print("Lignes à quantité 0 avec un volume > 0 :",
      int(((schedule["schedule_quantity"] == 0) & (schedule["volume"] > 0)).sum()))

Besoins le week-end : 0
Besoins le 11/11 (férié) : 108
Usines max par quai : 1
Articles livrés à plusieurs usines : 95 / 1110
Quantités entières : True
Article × quai dont le volume varie : 0 / 1303
Lignes à quantité 0 avec un volume > 0 : 891


**Les 24 cas atypiques du poids.** Pour chaque fournisseur × article × usine livré en au moins 2 quantités, on ajuste poids = part fixe + poids par pièce × quantité (même méthode que qualité §4). Une part fixe négative est impossible pour un emballage : ces cas sont conservés tels quels mais listés pour vérification.

In [5]:
cas = []
for (fournisseur, article, usine), g in besoins.dropna(subset=["weight"]).groupby(["supplier_id", "material", "plant"]):
    if g["schedule_quantity"].nunique() < 2:
        continue
    par_piece, fixe = np.polyfit(g["schedule_quantity"], g["weight"], 1)
    cas.append({"supplier_id": fournisseur, "material": article, "plant": usine,
                "quantites": g["schedule_quantity"].nunique(),
                "qte_min": g["schedule_quantity"].min(), "qte_max": g["schedule_quantity"].max(),
                "poids_min_kg": g["weight"].min(), "poids_max_kg": g["weight"].max(),
                "part_fixe_kg": fixe, "poids_par_piece_kg": par_piece})
cas = pd.DataFrame(cas)
atypiques = cas[cas["part_fixe_kg"] < -1].sort_values("part_fixe_kg")

print(f"Testés : {len(cas)} ; part fixe négative (< -1 kg) : {len(atypiques)} ; "
      f"dont 2 quantités seulement : {int((atypiques['quantites'] == 2).sum())} ; "
      f"par usine : {atypiques['plant'].value_counts().to_dict()}")
atypiques.round(3).reset_index(drop=True)

Testés : 327 ; part fixe négative (< -1 kg) : 24 ; dont 2 quantités seulement : 13 ; par usine : {'PLT_001': 15, 'PLT_004': 5, 'PLT_002': 4}


,supplier_id,material,plant,quantites,qte_min,qte_max,poids_min_kg,poids_max_kg,part_fixe_kg,poids_par_piece_kg
0,SUP_00149,MAT_000323,PLT_004,2,500.0,524.0,163.50,225.6,-1130.250,2.588
1,SUP_00144,MAT_000776,PLT_001,12,162.0,2240.0,305.90,5800.3,-304.798,2.589
2,SUP_00053,MAT_000668,PLT_001,4,2500.0,10000.0,25.35,211.8,-74.590,0.027
3,SUP_00083,MAT_000182,PLT_002,2,672.0,1548.0,1943.70,4570.1,-71.073,2.998
4,SUP_00118,MAT_001212,PLT_001,2,720.0,1640.0,179.80,483.1,-57.565,0.330
5,SUP_00118,MAT_001182,PLT_001,2,100.0,250.0,33.80,182.7,-49.100,0.829
6,SUP_00091,MAT_000657,PLT_001,2,1000.0,3000.0,4.29,96.6,-41.865,0.046
7,SUP_00112,MAT_001229,PLT_004,2,54.0,149.0,1593.50,4446.4,-28.148,30.031
8,SUP_00145,MAT_000817,PLT_004,2,160.0,300.0,340.30,657.3,-21.986,2.264
9,SUP_00107,MAT_000388,PLT_001,3,450.0,2500.0,12.11,145.1,-21.188,0.066


## 3.3 Dictionnaire de `distances.csv`

In [6]:
dist_planning = distances[distances["supplier_id"].isin(besoins["supplier_id"])]

pd.DataFrame([
    ("supplier_id", "–", "Fournisseur", "Clé unique"),
    ("distance", "km", "Distance fournisseur → point de consolidation (définition du cadrage)",
     "Entiers ; même distance pour toutes les usines d'un fournisseur ; manquants : qualité §7"),
], columns=["colonne", "unité", "description", "remarque"]).set_index("colonne").join(pd.DataFrame({
    "manquants (fichier)": distances.isna().sum(),
    "manquants (fournisseurs du planning)": dist_planning.isna().sum(),
    "distinctes": distances.nunique(),
    "min": distances.min(),
    "max": distances.max(),
}))

,unité,description,remarque,manquants (fichier),manquants (fournisseurs du planning),distinctes,min,max
colonne,,,,,,,,
supplier_id,–,Fournisseur,Clé unique,0,0,156,SUP_00001,SUP_00156
distance,km,Distance fournisseur → point de consolidation (définition du cadrage),Entiers ; même distance pour toutes les usines d'un fournisseur ; manquants : qualité §7,11,1,85,6.0,149.0


## 3.4 Dictionnaire de la grille tarifaire (`rate_sheet.pdf`)
Prix d'un envoi (= un camion) = coût de base × (1 + marge) × surcharge diesel. Lecture, contrôles et fonction `prix` : qualité §8.

In [7]:
pd.DataFrame([
    ("ligne", "km", "Tranche de distance, borne haute incluse (47 km → ligne 50)", f"{len(grille)} tranches, de {grille.index[0]} à {grille.index[-1]}"),
    ("colonne", "kg", "Tranche de poids d'un camion, borne haute incluse (316 kg → colonne 400)", f"{grille.shape[1]} tranches, de {grille.columns[0]} à {grille.columns[-1]}"),
    ("case", "EUR", "Coût de base d'un envoi", f"{grille.values.min()} à {grille.values.max()}"),
    ("marge", "%", "Marge appliquée au coût de base (PDF)", "239,23 %"),
    ("diesel", "facteur", "Surcharge carburant (PDF)", "1,1943"),
    ("validité", "date", "Date d'entrée en vigueur (PDF)", "01/08/2026"),
], columns=["élément", "unité", "description", "valeur"]).set_index("élément")

,unité,description,valeur
élément,,,
ligne,km,"Tranche de distance, borne haute incluse (47 km → ligne 50)","14 tranches, de 30 à >650"
colonne,kg,"Tranche de poids d'un camion, borne haute incluse (316 kg → colonne 400)","15 tranches, de 50 à >4000"
case,EUR,Coût de base d'un envoi,4.75 à 325.25
marge,%,Marge appliquée au coût de base (PDF),"239,23 %"
diesel,facteur,Surcharge carburant (PDF),"1,1943"
validité,date,Date d'entrée en vigueur (PDF),01/08/2026


Au-delà de 4 000 kg, le prix ne bouge plus jusqu'à 24,5 t (confirmé par le métier).

## 3.6 Données calculées

| Donnée | Formule | Unité | Calculée en |
|---|---|---|---|
| couple | `supplier_id` + `plant` | – | 3.6 |
| livraison actuelle | couple × date du besoin (11/11 férié → veille ouvrée) | – | 3.6 |
| poids / volume d'une livraison ou d'un camion | Σ `weight` / Σ `volume` des lignes regroupées | kg / m³ | 3.6 |
| nombre de camions | ⌈ max(poids / 24 500 ; volume / 110,4) ⌉ | camions | 3.6 (chargement exact : algorithme) |
| remplissage | poids / (camions × 24 500), idem en volume | % | 3.6 |
| coût d'un envoi (= un camion) | case(ligne km ≥ distance, colonne kg ≥ poids) × (1 + marge) × diesel | € | 3.6 |
| profil | fréquence (1, 2, 3, 4 ou 5 / semaine) + jours, identique chaque semaine | – | algorithme |
| date de livraison | dernier jour ouvré du profil ≤ date du besoin, écart ≤ 7 jours | date | algorithme |
| anticipation | date du besoin − date de livraison | jours | algorithme |
| économie | coût avant − coût après (et en % du coût avant) | € | algorithme |

 Chaque ligne a son propre emballage et son conteneur (confirmé) : la somme des poids et des volumes est la vraie charge d'un camion. Le calcul de ces données et leur preuve sur l'exemple du cadrage (SUP_00085, 1 215 €) sont dans le notebook qualité (§3.1 et §8).